# ARIMAX Baseline — Month-1 GDP Nowcasting

**Benchmark model for comparison with V16 GNN**

Uses the same data sources, same walk-forward protocol, same month-1 cutoff, and same SAAR evaluation as the V16 GNN notebook. The only difference is the model: **ARIMAX** (Auto-Regressive Integrated Moving Average with eXogenous variables) replaces the Graph Neural Network.

### Design

| Parameter | Value |
|-----------|-------|
| **Model** | ARIMAX (statsmodels SARIMAX) |
| **Target** | Quarterly GDP level |
| **Exogenous** | Top 5 features by |correlation| with GDP (per-fold, training-only) |
| **Data cutoff** | Month-1 (Jan/Apr/Jul/Oct) — ~45 day lead |
| **Order** | Auto-selected per fold via AIC grid search |
| **Walk-forward** | 80 folds (2006-Q1 to 2025-Q4) |
| **Evaluation** | SAAR MAE (same as V16 GNN) |

### Why ARIMAX as baseline?

- Classical time-series model — well understood, no deep learning
- Uses same month-1 data cutoff → apples-to-apples timing comparison
- Per-fold feature selection → no data leakage
- Simple, fast (~1 min total vs ~2.5 hours for GNN)

---
## Step 1 — Setup & Imports

In [ ]:
import warnings, os, sys, time, json, pathlib
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print(f"Python:  {sys.version.split()[0]}")
print(f"Pandas:  {pd.__version__}")
print(f"NumPy:   {np.__version__}")

# Check statsmodels
try:
    import statsmodels
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    print(f"Statsmodels: {statsmodels.__version__}")
except ImportError:
    raise ImportError("statsmodels required: pip install statsmodels")

---
## Step 2 — Paths & Parameters

In [ ]:
# ── Paths (same as V16 GNN) ──────────────────────────────────────
DATA_DIR   = pathlib.Path("./data")
FRED_DIR   = DATA_DIR / "input/new/ref_from_ST_FRED"
MRTS_DIR   = DATA_DIR / "input/new/raw_from_MRTS"
GDP_CSV    = FRED_DIR / "Quarterly_GDP.csv"
OUTPUT_DIR = DATA_DIR / "output/new/ARIMAX/quarterly_walkfwd_arimax"

import shutil
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"Cleared old results: {OUTPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

# ── Parameters ───────────────────────────────────────────────────
TEST_START = pd.Timestamp("2006-01-01")
TEST_END   = pd.Timestamp("2025-10-01")
MIN_TRAIN_Q = 16          # Minimum training quarters before first fold
N_EXOG      = 5           # Number of exogenous features (per-fold, by correlation)
LOOKBACK_Q  = 40          # Max quarterly observations for ARIMAX training

# ARIMAX order grid search (p, d, q)
P_RANGE = range(0, 4)     # AR order
D_RANGE = range(0, 2)     # Differencing
Q_RANGE = range(0, 3)     # MA order

def log(msg: str):
    print(msg, flush=True)

---
## Step 3 — Data Loading

Same loaders as V16 GNN — FRED monthly, MRTS monthly, GDP quarterly.

In [ ]:
def load_fred_monthly() -> pd.DataFrame:
    """Load all monthly FRED CSVs."""
    log("=" * 65)
    log("LOADING FRED MONTHLY DATA")
    log("=" * 65)
    dfs = []
    for fpath in sorted(FRED_DIR.glob("Monthly_*.csv")):
        try:
            df = pd.read_csv(fpath, parse_dates=["observation_date"])
            val_col = [c for c in df.columns if c != "observation_date"][0]
            df.rename(columns={"observation_date": "date"}, inplace=True)
            df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
            df = df.dropna(subset=[val_col]).set_index("date")[[val_col]]
            stem = fpath.stem.replace("Monthly_", "")
            parts = stem.split("_")
            col_name = f"F_{parts[0]}_{parts[1][:4]}" if len(parts) >= 2 else f"F_{parts[0]}"
            df.columns = [col_name]
            dfs.append(df)
            log(f"  ✓ {col_name:30s}  {len(df):4d} obs")
        except Exception as e:
            log(f"  ✗ {fpath.name}: {e}")
    merged = pd.concat(dfs, axis=1).sort_index().resample("MS").last()
    log(f"\n  FRED total: {merged.shape[1]} series, {merged.shape[0]} months")
    return merged

def load_mrts_monthly() -> pd.DataFrame:
    """Load monthly MRTS retail sales data."""
    log("\n" + "=" * 65)
    log("LOADING MRTS MONTHLY DATA")
    log("=" * 65)
    dfs = []
    for fpath in sorted(MRTS_DIR.glob("*.xlsx")):
        try:
            naics = fpath.name.split("_")[0]
            col_name = f"MRTS_{naics}"
            raw = pd.read_excel(fpath, header=None, skiprows=7)
            raw.columns = ["Period", "Value"] + [f"extra_{i}" for i in range(len(raw.columns) - 2)]
            raw = raw[["Period", "Value"]].copy()
            raw["Value"] = pd.to_numeric(raw["Value"], errors="coerce")
            raw = raw.dropna(subset=["Value"])
            raw = raw[raw["Period"].notna() & (raw["Period"] != "")]
            raw["date"] = pd.to_datetime(raw["Period"].astype(str), format="%b-%Y", errors="coerce")
            raw = raw.dropna(subset=["date"]).set_index("date")[["Value"]]
            raw.columns = [col_name]
            dfs.append(raw)
            log(f"  ✓ {col_name:15s}  {len(raw):4d} obs")
        except Exception as e:
            log(f"  ✗ {fpath.name}: {e}")
    if not dfs:
        return pd.DataFrame()
    merged = pd.concat(dfs, axis=1).sort_index().resample("MS").last()
    log(f"\n  MRTS total: {merged.shape[1]} series, {merged.shape[0]} months")
    return merged

def load_gdp_quarterly() -> pd.DataFrame:
    """Load quarterly GDP."""
    log("\n" + "=" * 65)
    log("LOADING GDP QUARTERLY DATA")
    log("=" * 65)
    df = pd.read_csv(GDP_CSV, parse_dates=["observation_date"])
    df.rename(columns={"observation_date": "date"}, inplace=True)
    df["GDP"] = pd.to_numeric(df["GDP"], errors="coerce")
    df = df.dropna(subset=["GDP"]).set_index("date")[["GDP"]]
    log(f"  {len(df)} quarters")
    return df

fred_df = load_fred_monthly()
mrts_df = load_mrts_monthly()
gdp_q   = load_gdp_quarterly()

---
## Step 4 — Merge Data

Combine FRED + MRTS + GDP into a single monthly DataFrame. Feature selection is deferred to per-fold (no leakage).

In [ ]:
def merge_all_data(fred_df, mrts_df, gdp_q):
    monthly_idx = pd.date_range(
        start=min(fred_df.index.min(), mrts_df.index.min() if len(mrts_df) else fred_df.index.min()),
        end=max(fred_df.index.max(), mrts_df.index.max() if len(mrts_df) else fred_df.index.max()),
        freq="MS",
    )
    gdp_monthly = gdp_q.reindex(monthly_idx).ffill()

    parts = [fred_df]
    if len(mrts_df) > 0:
        parts.append(mrts_df)
    parts.append(gdp_monthly)
    merged = pd.concat(parts, axis=1).sort_index()

    # Handle duplicate columns
    if merged.columns.duplicated().any():
        new_cols, seen = [], {}
        for c in merged.columns:
            if c in seen:
                seen[c] += 1
                new_cols.append(f"{c}_{seen[c]}")
            else:
                seen[c] = 0
                new_cols.append(c)
        merged.columns = new_cols

    all_cols = list(merged.columns)
    log("\n" + "=" * 65)
    log("MERGED MONTHLY DATASET")
    log("=" * 65)
    log(f"  Shape: {merged.shape}")
    log(f"  Date range: {merged.index.min().strftime('%Y-%m')} → {merged.index.max().strftime('%Y-%m')}")
    log(f"  Features: {len(all_cols) - 1}  (+GDP target)")
    return merged, all_cols

monthly_df, all_cols = merge_all_data(fred_df, mrts_df, gdp_q)

---
## Step 5 — Quarterly Aggregation with Month-1 Cutoff

ARIMAX works on quarterly data. For each quarter, we use only the **month-1** value (Jan/Apr/Jul/Oct) of each indicator — matching the V16 GNN's data cutoff exactly.

This means:
- Q1 prediction uses only January data
- Q2 prediction uses only April data
- etc.

The binding constraint is MRTS publication lag (~42 days), giving **~45 day lead** before GDP release.

In [ ]:
def build_quarterly_dataset(monthly_df: pd.DataFrame, gdp_q: pd.DataFrame) -> pd.DataFrame:
    """Convert monthly data to quarterly using month-1 values only.
    
    For each quarter, take the value from month-1:
      Q1 → January, Q2 → April, Q3 → July, Q4 → October
    """
    # Filter to month-1 rows only
    month1_months = [1, 4, 7, 10]
    month1_data = monthly_df[monthly_df.index.month.isin(month1_months)].copy()
    
    # Map to quarter-start dates
    month1_data.index = month1_data.index.to_period("Q").to_timestamp()
    
    # Replace forward-filled GDP with actual quarterly GDP
    # (The forward-filled GDP at month-1 contains current quarter's GDP = leakage!)
    # Use PREVIOUS quarter's GDP as the feature (most recent available at month-1)
    if "GDP" in month1_data.columns:
        month1_data["GDP_feature"] = month1_data["GDP"].shift(1)  # lag by 1 quarter
        month1_data = month1_data.drop(columns=["GDP"])
    
    # Add actual GDP as target (from quarterly source)
    month1_data = month1_data.join(gdp_q, how="left")
    
    log("\n" + "=" * 65)
    log("QUARTERLY DATASET (Month-1 Cutoff)")
    log("=" * 65)
    log(f"  Shape: {month1_data.shape}")
    log(f"  Date range: {month1_data.index.min().strftime('%Y-Q%q')} → {month1_data.index.max().strftime('%Y-Q%q')}")
    log(f"  Features: {month1_data.shape[1] - 1}  (GDP_feature = lagged GDP)")
    
    return month1_data

quarterly_df = build_quarterly_dataset(monthly_df, gdp_q)
print(f"\nFirst 5 rows:")
print(quarterly_df[["GDP", "GDP_feature"]].head())

---
## Step 6 — ARIMAX Model

For each fold:
1. **Feature selection**: Top 5 features by |correlation| with GDP on training data only
2. **Order selection**: Grid search over (p,d,q) orders, pick lowest AIC
3. **Fit**: Train ARIMAX on training quarters with selected exogenous features
4. **Predict**: One-step-ahead forecast for the test quarter

The model predicts GDP **level**, which is then converted to SAAR growth rate for evaluation (matching V16 GNN).

In [ ]:
def select_exog_features(train_df: pd.DataFrame, n_features: int = 5) -> List[str]:
    """Select top N features by |correlation| with GDP on training data only."""
    feature_cols = [c for c in train_df.columns if c not in ["GDP"]]
    valid_df = train_df[feature_cols + ["GDP"]].dropna()
    
    if len(valid_df) < 5:
        return feature_cols[:n_features]
    
    corrs = valid_df[feature_cols].corrwith(valid_df["GDP"]).abs()
    corrs = corrs.dropna().sort_values(ascending=False)
    return corrs.head(n_features).index.tolist()


def fit_arimax(train_gdp: pd.Series, train_exog: pd.DataFrame,
               p_range=range(0, 4), d_range=range(0, 2), q_range=range(0, 3)):
    """Fit ARIMAX with AIC-based order selection.
    
    Returns (fitted_model, best_order, best_aic).
    """
    best_aic = np.inf
    best_order = (1, 1, 0)
    best_model = None
    
    for p in p_range:
        for d in d_range:
            for q in q_range:
                if p == 0 and q == 0:
                    continue  # Skip trivial model
                try:
                    model = SARIMAX(
                        train_gdp,
                        exog=train_exog,
                        order=(p, d, q),
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    fitted = model.fit(disp=False, maxiter=200)
                    if fitted.aic < best_aic:
                        best_aic = fitted.aic
                        best_order = (p, d, q)
                        best_model = fitted
                except Exception:
                    continue
    
    # Fallback: simple AR(1) if nothing worked
    if best_model is None:
        try:
            model = SARIMAX(
                train_gdp,
                exog=train_exog,
                order=(1, 1, 0),
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            best_model = model.fit(disp=False, maxiter=200)
            best_order = (1, 1, 0)
            best_aic = best_model.aic
        except Exception:
            return None, (1, 1, 0), np.inf
    
    return best_model, best_order, best_aic


print("ARIMAX model functions defined.")

---
## Step 7 — Walk-Forward Evaluation

Same protocol as V16 GNN:
- Walk-forward from 2006-Q1 to 2025-Q4 (80 folds)
- Each fold trains on all available history before the test quarter
- Per-fold feature selection (no look-ahead bias)
- Evaluate on SAAR MAE

**Expected runtime**: ~1-2 minutes (vs ~2.5 hours for GNN).

In [ ]:
def fmt_q(dt: pd.Timestamp) -> str:
    return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"

def compute_saar(current_gdp: float, prev_gdp: float) -> float:
    """Compute SAAR growth rate from GDP levels."""
    if prev_gdp <= 0:
        return 0.0
    return ((current_gdp / prev_gdp) ** 4 - 1) * 100

In [ ]:
%%time
# ============================================================
# ARIMAX Walk-Forward Evaluation
# Same protocol as V16 GNN
# ============================================================

results_csv = OUTPUT_DIR / "walkforward_results.csv"

log("\n" + "=" * 65)
log("WALK-FORWARD EVALUATION — ARIMAX Baseline (Month-1 Cutoff)")
log("=" * 65)
log(f"  Exogenous features: Top {N_EXOG} by |correlation| (per-fold)")
log(f"  Order selection:    AIC grid search p={list(P_RANGE)}, d={list(D_RANGE)}, q={list(Q_RANGE)}")
log(f"  Data cutoff:        Month-1 (same as V16 GNN)")

test_dates = gdp_q.loc[TEST_START:TEST_END].index.tolist()
log(f"  Folds:              {len(test_dates)}")

results_list = []

for fold_i, test_q in enumerate(test_dates):
    t0 = time.time()
    
    # Split: training = all quarters before test_q
    train_mask = quarterly_df.index < test_q
    train_df = quarterly_df[train_mask].dropna(subset=["GDP"])
    
    if len(train_df) < MIN_TRAIN_Q:
        log(f"  Fold {fold_i:2d} | {fmt_q(test_q)} — skipped (only {len(train_df)} train quarters)")
        continue
    
    # Limit training window
    if len(train_df) > LOOKBACK_Q:
        train_df = train_df.iloc[-LOOKBACK_Q:]
    
    # Per-fold feature selection (leakage-free)
    exog_cols = select_exog_features(train_df, N_EXOG)
    
    # Prepare training data
    train_gdp = train_df["GDP"].copy()
    train_exog = train_df[exog_cols].copy()
    
    # Fill NaN in exogenous with forward-fill then backfill
    train_exog = train_exog.ffill().bfill().fillna(0)
    
    # Get test quarter exogenous
    if test_q not in quarterly_df.index:
        continue
    test_exog = quarterly_df.loc[[test_q], exog_cols].copy()
    test_exog = test_exog.ffill().bfill().fillna(0)
    
    # Fit ARIMAX
    fitted_model, best_order, best_aic = fit_arimax(
        train_gdp, train_exog, P_RANGE, D_RANGE, Q_RANGE
    )
    
    if fitted_model is None:
        log(f"  Fold {fold_i:2d} | {fmt_q(test_q)} — ARIMAX failed to fit")
        continue
    
    # Predict
    try:
        forecast = fitted_model.forecast(steps=1, exog=test_exog)
        pred_gdp = float(forecast.iloc[0])
    except Exception as e:
        log(f"  Fold {fold_i:2d} | {fmt_q(test_q)} — forecast failed: {e}")
        continue
    
    # Compute SAAR
    actual_gdp = float(gdp_q.loc[test_q, "GDP"])
    prev_q_dt = test_q - pd.DateOffset(months=3)
    if prev_q_dt in gdp_q.index:
        prev_gdp = float(gdp_q.loc[prev_q_dt, "GDP"])
    else:
        prev_qs = gdp_q.index[gdp_q.index < test_q]
        prev_gdp = float(gdp_q.loc[prev_qs[-1], "GDP"])
    
    actual_saar = compute_saar(actual_gdp, prev_gdp)
    pred_saar   = compute_saar(pred_gdp, prev_gdp)
    saar_error  = pred_saar - actual_saar
    saar_ae     = abs(saar_error)
    dir_ok      = (pred_saar >= 0) == (actual_saar >= 0)
    
    elapsed = time.time() - t0
    mark = "✓" if dir_ok else "✗"
    
    log(f"  Fold {fold_i:2d} | {fmt_q(test_q)} | Act={actual_saar:+7.2f}  "
        f"Pred={pred_saar:+7.2f} | AE={saar_ae:5.2f} | {mark} | "
        f"order={best_order} | {elapsed:.1f}s")
    
    row = {
        "fold": fold_i,
        "test_quarter": test_q.strftime("%Y-%m-%d"),
        "actual_gdp": actual_gdp,
        "pred_gdp": round(pred_gdp, 2),
        "prev_gdp": prev_gdp,
        "actual_saar": round(actual_saar, 4),
        "pred_saar": round(pred_saar, 4),
        "saar_error": round(saar_error, 4),
        "saar_abs_error": round(saar_ae, 4),
        "direction_correct": bool(dir_ok),
        "arimax_order": str(best_order),
        "aic": round(best_aic, 1),
        "n_train": len(train_df),
        "exog_features": ", ".join(exog_cols),
        "n_nodes": len(exog_cols) + 1,
        "elapsed_sec": round(elapsed, 2),
    }
    results_list.append(row)
    
    # Running stats
    running_mae = np.mean([r["saar_abs_error"] for r in results_list])
    running_dir = sum(1 for r in results_list if r["direction_correct"])
    log(f"         Running MAE: {running_mae:.3f} | Dir: {running_dir}/{len(results_list)}")

# Save results
rdf = pd.DataFrame(results_list)
rdf.to_csv(results_csv, index=False)

log(f"\n{'=' * 65}")
log(f"WALK-FORWARD COMPLETE — {len(results_list)} folds")
log(f"{'=' * 65}")

final_mae = rdf["saar_abs_error"].mean()
final_dir = rdf["direction_correct"].sum()
log(f"  Final MAE:       {final_mae:.3f} pp")
log(f"  Final Direction: {int(final_dir)}/{len(rdf)}")

# Save summary
summary = {
    "model": "ARIMAX_Month1_Baseline",
    "data_cutoff": "month-1 (Jan/Apr/Jul/Oct)",
    "timing_advantage": "~45 days before GDP advance estimate",
    "n_folds": len(rdf),
    "n_exog": N_EXOG,
    "saar_mae": round(float(final_mae), 4),
    "direction_accuracy": f"{int(final_dir)}/{len(rdf)}",
}
with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
log(f"  Saved to {OUTPUT_DIR}")

---
## Step 8 — Load Results

In [ ]:
# Load results for analysis
results = pd.read_csv(OUTPUT_DIR / "walkforward_results.csv")
results["test_quarter"] = pd.to_datetime(results["test_quarter"])
print(f"Loaded {len(results)} fold results")
print(f"Date range: {fmt_q(results['test_quarter'].min())} → {fmt_q(results['test_quarter'].max())}")
print(f"Overall MAE: {results['saar_abs_error'].mean():.3f} pp")
print(f"Direction:   {results['direction_correct'].sum()}/{len(results)}")

---
## Step 9 — Results Analysis

### 9a. Overall Performance

In [ ]:
print("=" * 60)
print("OVERALL PERFORMANCE — ARIMAX Baseline (Month-1)")
print("=" * 60)

mae     = results["saar_abs_error"].mean()
median  = results["saar_abs_error"].median()
std_ae  = results["saar_abs_error"].std()
signed  = results["saar_error"].mean()
dir_acc = results["direction_correct"].sum()
n       = len(results)

ctr_mask = results["actual_saar"] < 0
n_ctr    = int(ctr_mask.sum())
ctr_dir  = int(results.loc[ctr_mask, "direction_correct"].sum()) if n_ctr > 0 else 0

print(f"\n  SAAR MAE:              {mae:.3f} pp")
print(f"  SAAR Median AE:        {median:.3f} pp")
print(f"  SAAR Std AE:           {std_ae:.3f} pp")
print(f"  Signed Error (bias):   {signed:+.3f} pp")
print(f"  Direction Accuracy:    {dir_acc}/{n} ({100*dir_acc/n:.1f}%)")
print(f"  Contraction Direction: {ctr_dir}/{n_ctr}")
print(f"\n  Timing: ~45 days BEFORE GDP advance estimate")

### 9b. Recession vs Normal-Time Performance

In [ ]:
recession_qs = pd.to_datetime([
    "2008-07-01", "2008-10-01", "2009-01-01",   # GFC
    "2020-04-01", "2020-07-01",                   # COVID
])

rec_mask    = results["test_quarter"].isin(recession_qs)
normal_mask = ~rec_mask

rec    = results[rec_mask]
normal = results[normal_mask]

print("=" * 60)
print("RECESSION vs NORMAL-TIME PERFORMANCE")
print("=" * 60)
print(f"\n  Recession quarters ({len(rec)} folds):")
print(f"    MAE:       {rec['saar_abs_error'].mean():.3f} pp")
print(f"    Median AE: {rec['saar_abs_error'].median():.3f} pp")
print(f"    Direction: {rec['direction_correct'].sum()}/{len(rec)}")
for _, row in rec.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"      {q}: Act={row['actual_saar']:+7.2f}  Pred={row['pred_saar']:+7.2f}  AE={row['saar_abs_error']:5.2f}  {m}")

print(f"\n  Normal-time quarters ({len(normal)} folds):")
print(f"    MAE:       {normal['saar_abs_error'].mean():.3f} pp")
print(f"    Median AE: {normal['saar_abs_error'].median():.3f} pp")
print(f"    Direction: {normal['direction_correct'].sum()}/{len(normal)}")

### 9c. 2008 Financial Crisis — Detailed Breakdown

In [ ]:
print("=" * 60)
print("2008 FINANCIAL CRISIS — DETAILED BREAKDOWN")
print("=" * 60)

gfc_qs = pd.to_datetime(["2008-01-01", "2008-04-01", "2008-07-01", "2008-10-01", 
                          "2009-01-01", "2009-04-01", "2009-07-01", "2009-10-01"])
gfc = results[results["test_quarter"].isin(gfc_qs)].copy()

print(f"\n  {'Quarter':<12s} {'Actual':>10s} {'Predicted':>10s} {'Error':>10s} {'Abs Error':>10s} {'Dir':>5s}")
print(f"  {'─' * 60}")
for _, row in gfc.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"  {q:<12s} {row['actual_saar']:>+10.2f} {row['pred_saar']:>+10.2f} "
          f"{row['saar_error']:>+10.2f} {row['saar_abs_error']:>10.2f} {m:>5s}")

print(f"\n  GFC Period MAE: {gfc['saar_abs_error'].mean():.3f} pp")
print(f"  GFC Direction:  {gfc['direction_correct'].sum()}/{len(gfc)}")
print(f"\n  Note: The GFC featured unprecedented GDP drops. With only month-1")
print(f"  data, the model must predict these extreme swings ~45 days early.")

### 9d. COVID Period — Detailed Breakdown

In [ ]:
print("=" * 60)
print("COVID PERIOD — DETAILED BREAKDOWN")
print("=" * 60)

covid_qs = pd.to_datetime(["2020-01-01", "2020-04-01", "2020-07-01", "2020-10-01",
                            "2021-01-01", "2021-04-01"])
covid = results[results["test_quarter"].isin(covid_qs)].copy()

print(f"\n  {'Quarter':<12s} {'Actual':>10s} {'Predicted':>10s} {'Error':>10s} {'Abs Error':>10s} {'Dir':>5s}")
print(f"  {'─' * 60}")
for _, row in covid.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"  {q:<12s} {row['actual_saar']:>+10.2f} {row['pred_saar']:>+10.2f} "
          f"{row['saar_error']:>+10.2f} {row['saar_abs_error']:>10.2f} {m:>5s}")

print(f"\n  COVID Period MAE: {covid['saar_abs_error'].mean():.3f} pp")
print(f"  COVID Direction:  {covid['direction_correct'].sum()}/{len(covid)}")
print(f"\n  Key challenge: 2020-Q1 prediction uses only January 2020 data —")
print(f"  before COVID shutdowns began in March. The -29% Q2 crash required")
print(f"  predicting with only April data, before the full impact was visible.")

### 9e. Post-2011-Q3 Performance — GDPNow Comparison

In [ ]:
print("=" * 60)
print("POST-2011-Q3 PERFORMANCE (GDPNow Comparison Period)")
print("=" * 60)

post2011q3 = results[results["test_quarter"] >= "2011-07-01"]
p2011q3_rec  = post2011q3[post2011q3["test_quarter"].isin(recession_qs)]
p2011q3_norm = post2011q3[~post2011q3["test_quarter"].isin(recession_qs)]

print(f"  {'Period':<25s} {'N':>3s}  {'MAE':>7s}  {'Med AE':>7s}  {'Dir':>5s}")
print(f"  {'-'*25} {'---':>3s}  {'-----':>7s}  {'------':>7s}  {'---':>5s}")
print(f"  {'2011-Q3+ All':<25s} {len(post2011q3):3d}  {post2011q3['saar_abs_error'].mean():7.3f}  "
      f"{post2011q3['saar_abs_error'].median():7.3f}  "
      f"{post2011q3['direction_correct'].sum():>2d}/{len(post2011q3)}")
print(f"  {'2011-Q3+ Normal':<25s} {len(p2011q3_norm):3d}  {p2011q3_norm['saar_abs_error'].mean():7.3f}  "
      f"{p2011q3_norm['saar_abs_error'].median():7.3f}  "
      f"{p2011q3_norm['direction_correct'].sum():>2d}/{len(p2011q3_norm)}")
if len(p2011q3_rec) > 0:
    print(f"  {'2011-Q3+ Recession':<25s} {len(p2011q3_rec):3d}  {p2011q3_rec['saar_abs_error'].mean():7.3f}  "
          f"{p2011q3_rec['saar_abs_error'].median():7.3f}  "
          f"{p2011q3_rec['direction_correct'].sum():>2d}/{len(p2011q3_rec)}")

print(f"\n  GDPNow final MAE (normal):     ~0.8-1.2 pp")
print(f"  ARIMAX normal MAE (2011-Q3+):  {p2011q3_norm['saar_abs_error'].mean():.3f} pp")
print(f"  ARIMAX timing advantage:       ~30-45 days BEFORE GDP release")

### 9f. Performance by Era

In [ ]:
print("=" * 60)
print("PERFORMANCE BY ERA")
print("=" * 60)

eras = [
    ("Pre-GFC (2006–2007)",    "2006-01-01", "2007-12-31"),
    ("GFC (2008–2009)",        "2008-01-01", "2009-12-31"),
    ("Recovery (2010–2014)",   "2010-01-01", "2014-12-31"),
    ("Expansion (2015–2019)",  "2015-01-01", "2019-12-31"),
    ("COVID (2020)",           "2020-01-01", "2020-12-31"),
    ("Post-COVID (2021–2025)", "2021-01-01", "2025-12-31"),
]

print(f"\n  {'Era':<30s} {'Folds':>5s} {'MAE':>8s} {'MedAE':>8s} {'Dir':>8s} {'Bias':>8s}")
print(f"  {'─' * 70}")

for name, start, end in eras:
    mask = (results["test_quarter"] >= start) & (results["test_quarter"] <= end)
    era = results[mask]
    if len(era) == 0:
        continue

    era_mae  = era["saar_abs_error"].mean()
    era_med  = era["saar_abs_error"].median()
    era_dir  = f"{era['direction_correct'].sum()}/{len(era)}"
    era_bias = era["saar_error"].mean()

    print(f"  {name:<30s} {len(era):>5d} {era_mae:>7.3f}p {era_med:>7.3f}p {era_dir:>8s} {era_bias:>+7.3f}p")

overall_dir = f"{results['direction_correct'].sum()}/{len(results)}"

print(f"  {'─' * 70}")
print(f"  {'OVERALL':<30s} {len(results):>5d} {results['saar_abs_error'].mean():>7.3f}p "
      f"{results['saar_abs_error'].median():>7.3f}p "
      f"{overall_dir:>8s} "
      f"{results['saar_error'].mean():>+7.3f}p")

### 9g. Per-Quarter Detail with Prediction "As-Of" Date

For each quarter, we compute the **prediction as-of date** — when the prediction would actually be available, based on MRTS data lag (~42 days after month-end). We also compute how many **days ahead** of the GDP advance estimate this gives us.

- **as_of_date** = month-1 end + 42 days (MRTS publication lag, the binding constraint)
- **GDP release** = quarter end + 28 days (BEA advance estimate)
- **days_ahead** = GDP release − as_of_date

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)

detail = results.copy()

# Compute as-of date: month-1 end + 42 days MRTS lag
def compute_as_of_date(q):
    """Month-1 end date + 42 days MRTS publication lag."""
    month1 = pd.Timestamp(year=q.year, month=q.month, day=1)
    month1_end = month1 + pd.offsets.MonthEnd(0)
    return month1_end + pd.Timedelta(days=42)

def compute_gdp_release(q):
    """GDP advance estimate: ~28 days after quarter end."""
    qtr_end_month = {1: 3, 4: 6, 7: 9, 10: 12}[q.month]
    qtr_end = pd.Timestamp(year=q.year, month=qtr_end_month, day=1) + pd.offsets.MonthEnd(0)
    return qtr_end + pd.Timedelta(days=28)

detail["as_of_date"] = detail["test_quarter"].apply(compute_as_of_date)
detail["gdp_release"] = detail["test_quarter"].apply(compute_gdp_release)
detail["days_ahead"] = (detail["gdp_release"] - detail["as_of_date"]).dt.days

display_df = detail[['test_quarter', 'as_of_date', 'gdp_release', 'days_ahead',
                     'actual_saar', 'pred_saar', 'saar_abs_error', 
                     'direction_correct']].copy()

display_df['Quarter'] = display_df['test_quarter'].apply(fmt_q)
display_df['As-Of'] = display_df['as_of_date'].dt.strftime('%Y-%m-%d')
display_df['GDP Release'] = display_df['gdp_release'].dt.strftime('%Y-%m-%d')
display_df['Days Ahead'] = display_df['days_ahead']
display_df['Actual'] = display_df['actual_saar'].round(2)
display_df['Predicted'] = display_df['pred_saar'].round(2)
display_df['Abs Err'] = display_df['saar_abs_error'].round(2)
display_df['Dir'] = display_df['direction_correct'].map({True: '✓', False: '✗'})

print_df = display_df[['Quarter', 'As-Of', 'GDP Release', 'Days Ahead',
                        'Actual', 'Predicted', 'Abs Err', 'Dir']]

print(print_df.to_string(index=False))

print(f"\nAverage days ahead of GDP release: {detail['days_ahead'].mean():.0f} days")

### 9h. Error Distribution & Worst Predictions

In [ ]:
print("=" * 60)
print("ERROR DISTRIBUTION")
print("=" * 60)

ae = results["saar_abs_error"]
print(f"\n  Percentiles:")
for p in [25, 50, 75, 90, 95]:
    print(f"    {p}th:  {ae.quantile(p/100):.3f} pp")

print(f"\n  Within 1pp:  {(ae <= 1).sum()}/{len(ae)} ({100*(ae <= 1).mean():.0f}%)")
print(f"  Within 2pp:  {(ae <= 2).sum()}/{len(ae)} ({100*(ae <= 2).mean():.0f}%)")
print(f"  Within 3pp:  {(ae <= 3).sum()}/{len(ae)} ({100*(ae <= 3).mean():.0f}%)")
print(f"  Above 5pp:   {(ae > 5).sum()}/{len(ae)} ({100*(ae > 5).mean():.0f}%)")

print(f"\n{'=' * 60}")
print("WORST 10 PREDICTIONS")
print(f"{'=' * 60}")

worst = results.nlargest(10, "saar_abs_error")
print(f"\n  {'Quarter':<12s} {'Actual':>10s} {'Predicted':>10s} {'Abs Error':>10s} {'Dir':>5s}")
print(f"  {'─' * 50}")
for _, row in worst.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"  {q:<12s} {row['actual_saar']:>+10.2f} {row['pred_saar']:>+10.2f} {row['saar_abs_error']:>10.2f} {m:>5s}")

### 9i. COVID Negative Quarters (2020-Q1 & 2020-Q2)

The two quarters with **negative GDP growth** during COVID — 2020-Q1 (-3.28% SAAR) and 2020-Q2 (-29.11% SAAR). These represent the hardest prediction challenge: the model uses only January 2020 data (pre-shutdown) for Q1, and only April 2020 data for Q2.

In [ ]:
print("=" * 60)
print("COVID NEGATIVE QUARTERS — 2020-Q1 & 2020-Q2")
print("=" * 60)

covid_neg_qs = pd.to_datetime(["2020-01-01", "2020-04-01"])
covid_neg = results[results["test_quarter"].isin(covid_neg_qs)].copy()

print(f"\n  {'Quarter':<12s} {'Actual':>10s} {'Predicted':>10s} {'Error':>10s} {'Abs Error':>10s} {'Dir':>5s}")
print(f"  {'─' * 60}")
for _, row in covid_neg.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"  {q:<12s} {row['actual_saar']:>+10.2f} {row['pred_saar']:>+10.2f} "
          f"{row['saar_error']:>+10.2f} {row['saar_abs_error']:>10.2f} {m:>5s}")

print(f"\n  COVID Negative Quarters MAE: {covid_neg['saar_abs_error'].mean():.3f} pp")
print(f"  Direction Accuracy:          {covid_neg['direction_correct'].sum()}/{len(covid_neg)}")

print(f"\n  Context:")
print(f"    2020-Q1: Model uses only January 2020 data — before COVID shutdowns")
print(f"             began in mid-March. Actual GDP: -3.28% SAAR.")
print(f"    2020-Q2: Model uses only April 2020 data — partial shutdown data.")
print(f"             Actual GDP: -29.11% SAAR (worst quarter since records began).")

### 9j. 3-Panel Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 2, 1]})

dates = results["test_quarter"]

# Panel 1: Actual vs Predicted SAAR
ax = axes[0]
ax.plot(dates, results["actual_saar"], 'b-', linewidth=1.2, label="Actual SAAR", alpha=0.9)
ax.plot(dates, results["pred_saar"], 'r--', linewidth=1.0, label="ARIMAX Predicted", alpha=0.8)
ax.axhline(y=0, color='k', linewidth=0.5, linestyle='-')

for qs, qe, label in [("2008-01-01", "2009-07-01", "GFC"), 
                        ("2020-01-01", "2020-10-01", "COVID")]:
    ax.axvspan(pd.Timestamp(qs), pd.Timestamp(qe), alpha=0.1, color='gray')

ax.set_ylabel("SAAR Growth Rate (%)")
ax.set_title("ARIMAX Baseline — GDP SAAR Growth Rate (Actual vs Predicted)\n"
             f"MAE = {results['saar_abs_error'].mean():.3f}pp | "
             f"Direction = {results['direction_correct'].sum()}/{len(results)} | "
             f"Lead = ~45 days",
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Panel 2: Absolute Error
ax = axes[1]
colors = ['green' if e <= 2 else 'orange' if e <= 5 else 'red' for e in results["saar_abs_error"]]
ax.bar(dates, results["saar_abs_error"], color=colors, alpha=0.7, width=60)
ax.axhline(y=results["saar_abs_error"].mean(), color='red', linestyle='--', 
           label=f'Mean AE = {results["saar_abs_error"].mean():.2f}pp')
ax.axhline(y=results["saar_abs_error"].median(), color='blue', linestyle=':', 
           label=f'Median AE = {results["saar_abs_error"].median():.2f}pp')
ax.set_ylabel("Absolute Error (pp)")
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Panel 3: Direction Accuracy
ax = axes[2]
dir_colors = ['green' if d else 'red' for d in results["direction_correct"]]
ax.bar(dates, [1]*len(results), color=dir_colors, alpha=0.6, width=60)
ax.set_ylabel("Direction")
ax.set_yticks([0, 1])
ax.set_yticklabels(['Wrong', 'Correct'])
ax.set_xlabel("Quarter")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "arimax_results_3panel.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'arimax_results_3panel.png'}")

---
## Step 10 — Summary

In [ ]:
# Compute all summary metrics
post2011q3 = results[results["test_quarter"] >= "2011-07-01"]
post2011q3_normal = post2011q3[~post2011q3["test_quarter"].isin(recession_qs)]

rec    = results[results["test_quarter"].isin(recession_qs)]
normal = results[~results["test_quarter"].isin(recession_qs)]

gfc_qs_all = pd.to_datetime(["2008-01-01","2008-04-01","2008-07-01","2008-10-01",
                              "2009-01-01","2009-04-01","2009-07-01","2009-10-01"])
covid_qs_all = pd.to_datetime(["2020-01-01","2020-04-01","2020-07-01","2020-10-01",
                                "2021-01-01","2021-04-01"])
covid_neg_qs = pd.to_datetime(["2020-01-01", "2020-04-01"])

gfc_results = results[results["test_quarter"].isin(gfc_qs_all)]
covid_results = results[results["test_quarter"].isin(covid_qs_all)]
covid_neg_results = results[results["test_quarter"].isin(covid_neg_qs)]

print("╔" + "═" * 62 + "╗")
print("║  ARIMAX BASELINE — FINAL RESULTS SUMMARY                     ║")
print("║  Month-1 Cutoff · ~45 Days Before GDP Release                ║")
print("╚" + "═" * 62 + "╝")

print(f"\n  {'Metric':<45s} {'Value':>15s}")
print(f"  {'─' * 63}")
print(f"  {'Overall SAAR MAE':<45s} {results['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'Overall Median AE':<45s} {results['saar_abs_error'].median():>14.3f}pp")
print(f"  {'Direction Accuracy':<45s} {results['direction_correct'].sum():>3d}/{len(results)} ({100*results['direction_correct'].mean():.1f}%)")
print(f"  {'Signed Error (bias)':<45s} {results['saar_error'].mean():>+14.3f}pp")
print(f"  {'─' * 63}")
print(f"  {'Normal-Time MAE':<45s} {normal['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'Recession MAE':<45s} {rec['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'─' * 63}")
print(f"  {'GFC Period MAE (2008-2009)':<45s} {gfc_results['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'COVID Period MAE (2020-Q1 to 2021-Q2)':<45s} {covid_results['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'COVID Negative Quarters MAE (Q1+Q2 2020)':<45s} {covid_neg_results['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'─' * 63}")
print(f"  {'Post-2011-Q3 Overall MAE':<45s} {post2011q3['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'Post-2011-Q3 Normal-Time MAE':<45s} {post2011q3_normal['saar_abs_error'].mean():>14.3f}pp")
print(f"  {'─' * 63}")
print(f"  {'Prediction Lead Time':<45s} {'~45 days':>15s}")
print(f"  {'Data Cutoff':<45s} {'month-1':>15s}")
print(f"  {'Model':<45s} {'ARIMAX':>15s}")
print(f"  {'Exogenous Features':<45s} {N_EXOG:>15d}")
print(f"  {'Feature Selection':<45s} {'per-fold':>15s}")
print(f"  {'Walk-Forward Folds':<45s} {len(results):>15d}")

In [ ]:
gfc_negative_qs = pd.to_datetime(["2008-01-01", "2008-10-01", "2009-01-01", "2009-04-01"])
gfc_neg = results[results["test_quarter"].isin(gfc_negative_qs)].copy()

print("=" * 60)
print("GFC NEGATIVE QUARTERS")
print("=" * 60)
print(f"\n  {'Quarter':<12s} {'Actual':>10s} {'Predicted':>10s} {'Error':>10s} {'Abs Error':>10s} {'Dir':>5s}")
print(f"  {'─' * 60}")
for _, row in gfc_neg.iterrows():
    q = fmt_q(row['test_quarter'])
    m = '✓' if row['direction_correct'] else '✗'
    print(f"  {q:<12s} {row['actual_saar']:>+10.2f} {row['pred_saar']:>+10.2f} "
          f"{row['saar_error']:>+10.2f} {row['saar_abs_error']:>10.2f} {m:>5s}")

print(f"\n  GFC Negative Quarters MAE: {gfc_neg['saar_abs_error'].mean():.3f} pp")
print(f"  Direction Accuracy:        {gfc_neg['direction_correct'].sum()}/{len(gfc_neg)}")


---
## Methodology Notes

### Walk-Forward Protocol (identical to V16 GNN)
- **No future data leakage**: Each fold trains only on data before the test quarter
- **Per-fold feature selection**: Top 5 features by |correlation| on training data only
- **Per-fold scaling**: Not needed for ARIMAX (operates on raw GDP levels)
- **Month-1 cutoff**: Only first month of each quarter used

### ARIMAX Specifics
- **Order selection**: Grid search over p∈{0-3}, d∈{0-1}, q∈{0-2}, minimizing AIC
- **Exogenous variables**: Top 5 monthly indicators (sampled at month-1) by GDP correlation
- **GDP feature**: Lagged by 1 quarter (previous quarter's GDP, not current = target)
- **Prediction**: One-step-ahead forecast of GDP level, converted to SAAR
- **No ensemble**: Single model per fold (unlike GNN's 9-member trimmed mean ensemble)

### Evaluation Periods (identical to V16 GNN)
- **Overall**: All 80 folds (2006-Q1 to 2025-Q4)
- **Recession quarters**: 2008-Q3, 2008-Q4, 2009-Q1, 2020-Q2, 2020-Q3
- **GFC period**: 2008-Q1 through 2009-Q4 (8 quarters)
- **COVID period**: 2020-Q1 through 2021-Q2 (6 quarters)
- **COVID negative**: 2020-Q1, 2020-Q2 only
- **Post-2011-Q3**: GDPNow comparison period
- **Eras**: Pre-GFC, GFC, Recovery, Expansion, COVID, Post-COVID